# Análisis Exploratorio de Datos — Pozos Productores, Campo Rubiales

**Fuente:** `data/datos.csv` — inventario de pozos de crudo pesado del Campo Rubiales (Grupo Ecopetrol).

**Objetivo.** Entender la población de pozos antes de cualquier modelado: verificar la calidad del
registro, estandarizar el tipo y el dominio de cada variable, y caracterizar las relaciones que
gobiernan el desempeño del campo. Las preguntas que guían el recorrido son:

1. ¿Cómo se distribuyen las variables de yacimiento y de producción (corte de agua, caudal, gravedad API, profundidad)?
2. ¿El **corte de agua** cambia según el **método de extracción**, o el método es consecuencia de la madurez del pozo?
3. ¿Qué variables se **correlacionan** entre sí y cuáles no aportan información?
4. ¿La elección del **método de extracción** siguió una tendencia según el **año de inicio de operación**?
5. ¿La **gravedad API** segrega por **zona operativa**?

**Ruta del análisis:** carga → auditoría de calidad → limpieza y estandarización → univariado →
bivariado → multivariado → hallazgos.

## 1. Configuración del entorno

Las dependencias son las mismas del proyecto (`requirements.txt`): `pandas` y `plotly`. Todos los
gráficos son interactivos: se puede hacer hover sobre cualquier marca para leer sus valores, aislar
series desde la leyenda y hacer zoom.

In [32]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("pandas:", pd.__version__)

pandas: 3.0.5


### Paleta y plantilla visual

Un único lugar define colores, tipografía y rejilla, para que todas las figuras del notebook se lean
como un mismo sistema. La paleta categórica se usa **en orden fijo** (un método de extracción conserva
su color en todas las figuras) y nunca se cicla; la matriz de correlaciones usa una escala divergente
azul ↔ rojo con gris neutro en el cero, porque el signo de la correlación es la información principal.

In [33]:
# --- Tokens de color (paleta validada, modo claro) -------------------------
SURFACE = "#fcfcfb"      # fondo del área de graficación
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECOND = "#52514e"
GRID = "#e8e7e3"

CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # slots categóricos 1-4
SEQ = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]  # rampa secuencial (magnitud)
DIVERGING = [[0.0, "#104281"], [0.5, "#f0efec"], [1.0, "#8c1c1c"]]  # azul ↔ gris ↔ rojo

# --- Plantilla base --------------------------------------------------------
pio.templates["rubiales"] = go.layout.Template(
    layout=dict(
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Helvetica, Arial, sans-serif", size=13, color=TEXT_SECOND),
        title=dict(font=dict(size=18, color=TEXT_PRIMARY), x=0.01, xanchor="left",
                   y=0.97, yanchor="top"),
        colorway=CAT,
        xaxis=dict(showgrid=False, linecolor=GRID, ticks="outside", tickcolor=GRID,
                   title=dict(font=dict(size=12, color=TEXT_SECOND))),
        yaxis=dict(gridcolor=GRID, zeroline=False, linecolor=SURFACE,
                   title=dict(font=dict(size=12, color=TEXT_SECOND))),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
                    bgcolor="rgba(0,0,0,0)", title=dict(text="")),
        margin=dict(l=70, r=30, t=90, b=60),
        hoverlabel=dict(bgcolor=SURFACE, font=dict(color=TEXT_PRIMARY, size=12),
                        bordercolor=GRID),
    )
)
pio.templates.default = "rubiales"


# Encabezado estandar: titulo a la izquierda y una linea de lectura opcional debajo.
def titular(fig, titulo, subtitulo=None, alto=460):
    texto = titulo if subtitulo is None else f"{titulo}<br><span style='font-size:13px;color:{TEXT_SECOND}'>{subtitulo}</span>"
    # La leyenda se ancla justo encima del area de graficacion: el margen superior
    # debe dejar espacio para el titulo, el subtitulo y la propia leyenda.
    fig.update_layout(title=dict(text=texto), height=alto,
                      margin_t=150 if subtitulo else 110)
    return fig

## 2. Carga de los datos

La ruta se resuelve de forma relativa al notebook (`notebooks/` → `../data/datos.csv`), con respaldo
por si el kernel arranca en la raíz del proyecto.

In [34]:
def ruta_datos(nombre: str = "datos.csv") -> Path:
    candidatas = [Path("..") / "data" / nombre, Path("data") / nombre, Path.cwd().parent / "data" / nombre]
    for c in candidatas:
        if c.exists():
            return c.resolve()
    raise FileNotFoundError(f"No se encontró {nombre} en: {[str(c) for c in candidatas]}")


RUTA = ruta_datos()
crudo = pd.read_csv(RUTA)

print(f"Archivo : {RUTA}")
print(f"Registros: {crudo.shape[0]}  |  Columnas: {crudo.shape[1]}")
crudo.head()

Archivo : D:\developer_2026\sesion_3_sprint_7_21_07_2026\streamlit_app_v210726\data\datos.csv
Registros: 250  |  Columnas: 16


,id_pozo,campo,zona_operativa,metodo_extraccion,estado_operativo,fecha_inicio_operacion,fecha_ultimo_reporte,profundidad_m,produccion_bpd,produccion_acumulada_mbbl,corte_agua_pct,api_gravedad,presion_cabeza_psi,temperatura_fondo_c,activo,requiere_mantenimiento
0,RUB-001,Campo Rubiales,Occidente,Gas Lift,En Mantenimiento,2018-01-01,2026-07-04,2604.1,0.1,1.73,34.9,11.6,228.9,74.0,False,True
1,RUB-002,Campo Rubiales,Oriente,Bombeo Mecanico,Activo,2013-06-19,2026-07-13,2045.3,51.5,1.81,52.3,12.1,182.7,71.1,True,True
2,RUB-003,Campo Rubiales,Occidente,Bombeo Mecanico,En Mantenimiento,2017-10-19,2026-07-08,2758.9,11.5,1.03,19.0,10.2,231.2,70.3,False,True
3,RUB-004,Campo Rubiales,Norte,Bombeo Mecanico,En Mantenimiento,2023-07-03,2026-07-01,1922.3,3.6,0.31,21.2,11.1,281.8,92.0,False,True
4,RUB-005,Campo Rubiales,Oriente,Flujo Natural,En Mantenimiento,2018-03-27,2026-07-07,2439.1,3.4,0.52,38.7,13.0,247.8,74.7,False,True


In [35]:
crudo.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id_pozo                    250 non-null    str    
 1   campo                      250 non-null    str    
 2   zona_operativa             250 non-null    str    
 3   metodo_extraccion          250 non-null    str    
 4   estado_operativo           250 non-null    str    
 5   fecha_inicio_operacion     250 non-null    str    
 6   fecha_ultimo_reporte       250 non-null    str    
 7   profundidad_m              250 non-null    float64
 8   produccion_bpd             250 non-null    float64
 9   produccion_acumulada_mbbl  250 non-null    float64
 10  corte_agua_pct             250 non-null    float64
 11  api_gravedad               250 non-null    float64
 12  presion_cabeza_psi         250 non-null    float64
 13  temperatura_fondo_c        250 non-null    float64
 14  activ

### Diccionario de variables

| Columna | Tipo | Significado |
|---|---|---|
| `id_pozo` | texto | Identificador único del pozo (`RUB-###`) |
| `campo` | categórica | Campo al que pertenece el pozo |
| `zona_operativa` | categórica | Sector del campo: Norte, Sur, Oriente, Occidente, Centro |
| `metodo_extraccion` | categórica | Sistema de levantamiento: Flujo Natural, Bombeo Mecánico, Gas Lift, BES (bomba electrosumergible) |
| `estado_operativo` | categórica | Activo, En Mantenimiento, Cerrado |
| `fecha_inicio_operacion` | fecha | Puesta en producción del pozo |
| `fecha_ultimo_reporte` | fecha | Fecha del último reporte operativo |
| `profundidad_m` | numérica | Profundidad medida, en metros |
| `produccion_bpd` | numérica | Producción de crudo, barriles por día |
| `produccion_acumulada_mbbl` | numérica | Producción acumulada, millones de barriles |
| `corte_agua_pct` | numérica | Fracción de agua en el fluido producido, % |
| `api_gravedad` | numérica | Gravedad API del crudo (a menor valor, crudo más pesado) |
| `presion_cabeza_psi` | numérica | Presión en cabeza de pozo, psi |
| `temperatura_fondo_c` | numérica | Temperatura de fondo, °C |
| `activo` | booleana | Bandera de pozo en producción |
| `requiere_mantenimiento` | booleana | Bandera de intervención pendiente |

## 3. Auditoría de calidad

Antes de graficar nada conviene responder cinco preguntas: ¿faltan valores?, ¿hay pozos repetidos?,
¿alguna columna es constante (y por tanto inútil)?, ¿los valores caen en rangos físicamente posibles?,
¿las columnas redundantes se contradicen entre sí?

In [36]:
resumen = pd.DataFrame({
    "tipo": crudo.dtypes.astype(str),
    "nulos": crudo.isna().sum(),
    "% nulos": (crudo.isna().mean() * 100).round(2),
    "valores_unicos": crudo.nunique(),
})
resumen["ejemplo"] = [crudo[c].dropna().iloc[0] if crudo[c].notna().any() else None for c in crudo.columns]
resumen

,tipo,nulos,% nulos,valores_unicos,ejemplo
id_pozo,str,0,0.0,250,RUB-001
campo,str,0,0.0,1,Campo Rubiales
zona_operativa,str,0,0.0,5,Occidente
metodo_extraccion,str,0,0.0,4,Gas Lift
estado_operativo,str,0,0.0,3,En Mantenimiento
fecha_inicio_operacion,str,0,0.0,248,2018-01-01
fecha_ultimo_reporte,str,0,0.0,30,2026-07-04
profundidad_m,float64,0,0.0,247,2604.1
produccion_bpd,float64,0,0.0,223,0.1
produccion_acumulada_mbbl,float64,0,0.0,166,1.73


In [37]:
print("Filas duplicadas completas :", crudo.duplicated().sum())
print("IDs de pozo duplicados     :", crudo["id_pozo"].duplicated().sum())

constantes = [c for c in crudo.columns if crudo[c].nunique(dropna=False) <= 1]
print("Columnas constantes        :", constantes)

print("\nDominio de las variables categóricas:")
for col in ["campo", "zona_operativa", "metodo_extraccion", "estado_operativo"]:
    print(f"  {col:20s} -> {sorted(crudo[col].unique())}")

Filas duplicadas completas : 0
IDs de pozo duplicados     : 0
Columnas constantes        : ['campo']

Dominio de las variables categóricas:
  campo                -> ['Campo Rubiales']
  zona_operativa       -> ['Centro', 'Norte', 'Occidente', 'Oriente', 'Sur']
  metodo_extraccion    -> ['BES', 'Bombeo Mecanico', 'Flujo Natural', 'Gas Lift']
  estado_operativo     -> ['Activo', 'Cerrado', 'En Mantenimiento']


In [38]:
# Rango físico de las variables numéricas
crudo.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
profundidad_m,250.0,2419.67,353.87,1570.80,2146.32,2415.60,2662.35,3457.40
produccion_bpd,250.0,119.14,112.87,0.00,25.08,89.40,179.40,462.20
produccion_acumulada_mbbl,250.0,1.55,0.98,0.31,0.83,1.36,2.01,6.15
corte_agua_pct,250.0,45.54,16.33,10.40,33.80,46.45,58.28,79.10
api_gravedad,250.0,12.39,2.20,7.50,10.90,12.50,13.98,18.70
presion_cabeza_psi,250.0,211.41,72.45,0.30,194.10,225.80,252.20,315.60
temperatura_fondo_c,250.0,81.49,9.41,64.20,74.05,81.85,89.38,96.80


In [39]:
# Coherencia entre columnas redundantes y ceros estructurales
print("Tabla de contingencia estado_operativo x activo")
print(pd.crosstab(crudo["estado_operativo"], crudo["activo"]), "\n")

print("Producción por estado operativo (bpd)")
print(crudo.groupby("estado_operativo")["produccion_bpd"].describe()[["count", "mean", "min", "max"]].round(2), "\n")

print("Pozos con produccion_bpd == 0 :", int((crudo["produccion_bpd"] == 0).sum()))
print("Rango de gravedad API         :", crudo["api_gravedad"].min(), "-", crudo["api_gravedad"].max())
print("Rango de corte de agua (%)    :", crudo["corte_agua_pct"].min(), "-", crudo["corte_agua_pct"].max())

Tabla de contingencia estado_operativo x activo
activo            False  True 
estado_operativo              
Activo                0    184
Cerrado              21      0
En Mantenimiento     45      0 

Producción por estado operativo (bpd)
                  count    mean  min    max
estado_operativo                           
Activo            184.0  158.42  4.8  462.2
Cerrado            21.0    0.00  0.0    0.0
En Mantenimiento   45.0   14.16  0.1   42.2 

Pozos con produccion_bpd == 0 : 21
Rango de gravedad API         : 7.5 - 18.7
Rango de corte de agua (%)    : 10.4 - 79.1


**Lectura de la auditoría.**

- **No hay valores faltantes ni pozos duplicados**: las 250 filas corresponden a 250 pozos distintos.
- `campo` es **constante** (`Campo Rubiales`): no aporta variación y sale del análisis, aunque se
  conserva en el dataset por trazabilidad.
- `activo` es **redundante** con `estado_operativo` (`Activo` ⇔ `True`, sin una sola contradicción):
  se puede reconstruir en cualquier momento, así que no se usa como variable independiente.
- Los **21 pozos con producción 0 bpd son exactamente los cerrados**: son ceros *estructurales*
  (el pozo no produce porque está fuera de servicio), no datos faltantes disfrazados. Por eso **no se
  imputan ni se eliminan**, pero sí se excluyen de los análisis de desempeño productivo, donde
  arrastrarían todas las medias hacia abajo.
- Todos los valores caen en rangos físicamente plausibles: la gravedad API va de 7.5 a 18.7, es decir
  **todo el campo es crudo pesado y extrapesado** (API < 22.3), lo que concuerda con la naturaleza de
  Rubiales; el corte de agua nunca excede 100 %; las profundidades están entre ~1 500 y ~3 500 m.

Conclusión: el dataset está **limpio**. La etapa siguiente no es de reparación sino de
**estandarización y enriquecimiento**.

## 4. Limpieza, estandarización y variables derivadas

Cuatro operaciones, cada una con su motivo:

1. **Fechas a `datetime`** — sin esto no se puede calcular antigüedad ni agrupar por cohorte.
2. **Normalización de texto** — se recortan espacios y se unifican mayúsculas en las categóricas, de
   modo que `"BES "` y `"BES"` no se cuenten como dos métodos distintos.
3. **Categóricas ordenadas** — el método de extracción y la zona se ordenan por una magnitud con
   sentido físico (corte de agua mediano y API mediana), para que los ejes de todas las figuras se
   lean de menor a mayor en lugar de en orden alfabético.
4. **Variables derivadas** — la antigüedad, la cohorte de arranque y una clasificación del corte de
   agua convierten información que estaba implícita en las fechas en variables analizables.

In [40]:
df = crudo.copy()

# 1) Fechas
for col in ["fecha_inicio_operacion", "fecha_ultimo_reporte"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
print("Fechas no parseables:", df[["fecha_inicio_operacion", "fecha_ultimo_reporte"]].isna().sum().sum())

# 2) Texto de las categóricas
for col in ["campo", "zona_operativa", "metodo_extraccion", "estado_operativo"]:
    df[col] = df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

# 3) Variables derivadas
CORTE = df["fecha_ultimo_reporte"].max()          # fecha de corte del inventario
df["anios_operacion"] = ((CORTE - df["fecha_inicio_operacion"]).dt.days / 365.25).round(2)
df["anio_inicio"] = df["fecha_inicio_operacion"].dt.year
df["cohorte_inicio"] = pd.cut(
    df["anio_inicio"],
    bins=[2007, 2011, 2015, 2019, 2024],
    labels=["2008-2011", "2012-2015", "2016-2019", "2020-2023"],
)
df["dias_sin_reporte"] = (CORTE - df["fecha_ultimo_reporte"]).dt.days
df["nivel_corte_agua"] = pd.cut(
    df["corte_agua_pct"],
    bins=[-0.01, 30, 50, 70, 100],
    labels=["Bajo (<30%)", "Medio (30-50%)", "Alto (50-70%)", "Muy alto (>70%)"],
)

# 4) Categóricas ordenadas por magnitud
orden_metodo = df.groupby("metodo_extraccion")["corte_agua_pct"].median().sort_values().index.tolist()
orden_zona = df.groupby("zona_operativa")["api_gravedad"].median().sort_values().index.tolist()
orden_estado = ["Activo", "En Mantenimiento", "Cerrado"]

df["metodo_extraccion"] = pd.Categorical(df["metodo_extraccion"], categories=orden_metodo, ordered=True)
df["zona_operativa"] = pd.Categorical(df["zona_operativa"], categories=orden_zona, ordered=True)
df["estado_operativo"] = pd.Categorical(df["estado_operativo"], categories=orden_estado, ordered=True)

print(f"Fecha de corte del inventario: {CORTE.date()}")
print("Orden de métodos (por corte de agua mediano):", orden_metodo)
print("Orden de zonas  (por API mediana)           :", orden_zona)
df[["id_pozo", "anios_operacion", "cohorte_inicio", "nivel_corte_agua", "dias_sin_reporte"]].head()

Fechas no parseables: 0
Fecha de corte del inventario: 2026-07-21
Orden de métodos (por corte de agua mediano): ['BES', 'Gas Lift', 'Bombeo Mecanico', 'Flujo Natural']
Orden de zonas  (por API mediana)           : ['Centro', 'Norte', 'Occidente', 'Sur', 'Oriente']


,id_pozo,anios_operacion,cohorte_inicio,nivel_corte_agua,dias_sin_reporte
0,RUB-001,8.55,2016-2019,Medio (30-50%),17
1,RUB-002,13.09,2012-2015,Alto (50-70%),8
2,RUB-003,8.75,2016-2019,Bajo (<30%),13
3,RUB-004,3.05,2020-2023,Bajo (<30%),20
4,RUB-005,8.32,2016-2019,Medio (30-50%),14


In [41]:
# Color fijo por método: la misma entidad conserva su color en todo el notebook
COLOR_METODO = {m: CAT[i] for i, m in enumerate(orden_metodo)}
COLOR_ESTADO = dict(zip(orden_estado, CAT[:3]))

# Subconjunto de pozos que efectivamente producen: base de los análisis de desempeño
activos = df[df["estado_operativo"] == "Activo"].copy()

print("Pozos totales :", len(df))
print("Pozos activos :", len(activos))
COLOR_METODO

Pozos totales : 250
Pozos activos : 184


{'BES': '#2a78d6',
 'Gas Lift': '#eb6834',
 'Bombeo Mecanico': '#1baf7a',
 'Flujo Natural': '#eda100'}

### Tarjetas de contexto

Cinco cifras que enmarcan todo lo que sigue.

In [42]:
kpis = pd.DataFrame([
    ("Pozos inventariados",        f"{len(df)}"),
    ("Pozos activos",              f"{len(activos)} ({len(activos)/len(df):.0%})"),
    ("Producción total del campo", f"{df['produccion_bpd'].sum():,.0f} bpd"),
    ("Corte de agua mediano",      f"{df['corte_agua_pct'].median():.1f} %"),
    ("Gravedad API mediana",       f"{df['api_gravedad'].median():.1f} °API (crudo pesado)"),
    ("Antigüedad mediana",         f"{df['anios_operacion'].median():.1f} años"),
], columns=["Indicador", "Valor"])
kpis.style.hide(axis="index")

Indicador,Valor
Pozos inventariados,250
Pozos activos,184 (74%)
Producción total del campo,"29,786 bpd"
Corte de agua mediano,46.5 %
Gravedad API mediana,12.5 °API (crudo pesado)
Antigüedad mediana,10.6 años


## 5. Análisis univariado — ¿cómo se distribuye cada variable?

Los histogramas responden a una pregunta previa a cualquier correlación: **qué forma tiene cada
variable**. Una distribución simétrica, una asimétrica con cola larga y una bimodal exigen lecturas
distintas y toleran estadísticos distintos (la media miente en las dos últimas).

In [43]:
variables = [
    ("corte_agua_pct", "Corte de agua (%)"),
    ("produccion_bpd", "Producción (bpd)"),
    ("api_gravedad", "Gravedad API (°API)"),
    ("profundidad_m", "Profundidad (m)"),
]

titulos = [f"{t}  ·  mediana {df[c].median():,.1f}" for c, t in variables]
fig = make_subplots(rows=2, cols=2, subplot_titles=titulos,
                    horizontal_spacing=0.12, vertical_spacing=0.20)

for i, (col, etiqueta) in enumerate(variables):
    r, c = divmod(i, 2)
    fig.add_trace(
        go.Histogram(
            x=df[col], nbinsx=28, marker=dict(color=CAT[0], line=dict(color=SURFACE, width=1)),
            name=etiqueta, showlegend=False,
            hovertemplate=f"{etiqueta}: %{{x}}<br>Pozos: %{{y}}<extra></extra>",
        ),
        row=r + 1, col=c + 1,
    )
    fig.add_vline(x=df[col].median(), line=dict(color=TEXT_SECOND, width=1, dash="dot"),
                  row=r + 1, col=c + 1)

fig.update_annotations(font=dict(size=13, color=TEXT_PRIMARY))
fig.update_yaxes(title_text="Nº de pozos", col=1)
fig.update_layout(bargap=0.04)
titular(fig, "Distribución de las cuatro variables de estado del pozo",
        "250 pozos · la línea punteada marca la mediana", alto=680)
fig.show()

**Lectura.**

- **Corte de agua**: distribución ancha y prácticamente uniforme entre 10 % y 79 %. No hay un valor
  típico de campo: cada pozo está en un punto distinto de su curva de agotamiento. Esta dispersión es
  precisamente lo que vale la pena explicar.
- **Producción**: fuerte asimetría a la derecha con una acumulación en cero. Los ceros son los pozos
  cerrados y los valores bajos los que están en mantenimiento; la cola derecha son unos pocos pozos de
  alto caudal que concentran buena parte del total. La **mediana** es aquí mucho más representativa que
  la media.
- **Gravedad API**: la única variable con forma cercana a la normal, centrada en ~12.5 °API y
  **enteramente por debajo de 20**: todo el crudo del campo es pesado. La distribución es ligeramente
  ancha porque, como se verá, mezcla cinco zonas con calidades distintas.
- **Profundidad**: simétrica alrededor de ~2 400 m, sin colas ni valores atípicos. Todos los pozos
  atacan el mismo intervalo productor.

In [44]:
fig = go.Figure()
conteo = df["nivel_corte_agua"].value_counts().reindex(df["nivel_corte_agua"].cat.categories)

fig.add_trace(go.Bar(
    x=conteo.index.astype(str), y=conteo.values,
    marker=dict(color=SEQ[1:5], line=dict(color=SURFACE, width=2)),
    text=[f"{v} pozos<br>{v/len(df):.0%}" for v in conteo.values],
    textposition="outside", textfont=dict(color=TEXT_SECOND, size=12),
    hovertemplate="%{x}<br>%{y} pozos<extra></extra>", showlegend=False,
))
fig.update_yaxes(title_text="Nº de pozos", range=[0, conteo.max() * 1.25])
titular(fig, "Madurez del campo según corte de agua",
        "42 % del inventario ya supera el 50 % de agua en el fluido producido · mediana del campo: 46.5 %")
fig.show()

## 6. Corte de agua y método de extracción

Esta es la relación por la que suele empezar cualquier revisión de un campo maduro. El diagrama de
cajas muestra la distribución completa por método —no solo el promedio—, con los pozos individuales
superpuestos para que se vea el tamaño de cada grupo y su dispersión real.

In [45]:
fig = go.Figure()
for metodo in orden_metodo:
    sub = df[df["metodo_extraccion"] == metodo]
    fig.add_trace(go.Box(
        y=sub["corte_agua_pct"], name=metodo,
        marker=dict(color=COLOR_METODO[metodo], size=5, opacity=0.55,
                    line=dict(color=SURFACE, width=1)),
        line=dict(width=2), fillcolor="rgba(0,0,0,0)",
        boxpoints="all", jitter=0.45, pointpos=0, width=0.5,
        hovertemplate="%{x}<br>Corte de agua: %{y:.1f}%<extra></extra>",
    ))

fig.update_yaxes(title_text="Corte de agua (%)")
fig.update_xaxes(title_text="Método de extracción")
fig.update_layout(showlegend=False)
titular(fig, "Corte de agua por método de extracción",
        "Cada punto es un pozo · métodos ordenados por corte de agua mediano", alto=520)
fig.show()

In [46]:
tabla = (df.groupby("metodo_extraccion", observed=True)
           .agg(pozos=("id_pozo", "count"),
                corte_agua_medio=("corte_agua_pct", "mean"),
                corte_agua_mediano=("corte_agua_pct", "median"),
                produccion_mediana=("produccion_bpd", "median"),
                antiguedad_mediana=("anios_operacion", "median"))
           .round(1))
tabla

,pozos,corte_agua_medio,corte_agua_mediano,produccion_mediana,antiguedad_mediana
metodo_extraccion,,,,,
BES,71,39.6,40.5,263.9,7.5
Gas Lift,37,42.9,44.4,138.8,9.7
Bombeo Mecanico,100,47.0,50.0,88.1,11.3
Flujo Natural,42,54.6,55.4,43.2,14.8


**Lectura — y una advertencia sobre la causalidad.**

El gradiente es nítido: la mediana de corte de agua va de **~55 % en Flujo Natural** a **50 % en Bombeo
Mecánico**, **~44 % en Gas Lift** y **~40 % en BES**. La tentación es concluir que el sistema de levantamiento
"causa" el corte de agua, pero la última columna de la tabla desmiente esa lectura: los pozos de
Flujo Natural tienen una **antigüedad mediana de ~15 años** y los de BES de ~7.5. La variable que
realmente ordena el grupo es la **madurez del pozo**; el método de extracción está *correlacionado*
con ella porque el campo fue cambiando de tecnología con el tiempo. El siguiente gráfico separa ambos
efectos.

## 7. Dispersión — antigüedad, agua y producción

Tres preguntas encadenadas: ¿el agua entra con los años?, ¿el agua se come la producción?, ¿el método
de extracción altera esa relación o solo ocupa un tramo distinto de la misma curva?

In [47]:
fig = make_subplots(rows=1, cols=4, shared_yaxes=True, horizontal_spacing=0.02,
                    subplot_titles=orden_metodo)

for i, metodo in enumerate(orden_metodo):
    sub = df[df["metodo_extraccion"] == metodo]
    fig.add_trace(go.Scatter(
        x=sub["anios_operacion"], y=sub["corte_agua_pct"], mode="markers",
        marker=dict(color=COLOR_METODO[metodo], size=9, opacity=0.75,
                    line=dict(color=SURFACE, width=1.5)),
        name=metodo, showlegend=False, customdata=sub[["id_pozo", "produccion_bpd"]],
        hovertemplate=("%{customdata[0]}<br>Antigüedad: %{x:.1f} años"
                       "<br>Corte de agua: %{y:.1f}%<br>Producción: %{customdata[1]:.1f} bpd<extra></extra>"),
    ), row=1, col=i + 1)

    # Recta de tendencia del propio grupo
    m, b = np.polyfit(sub["anios_operacion"], sub["corte_agua_pct"], 1)
    xs = np.array([sub["anios_operacion"].min(), sub["anios_operacion"].max()])
    r = sub["anios_operacion"].corr(sub["corte_agua_pct"])
    fig.add_trace(go.Scatter(
        x=xs, y=m * xs + b, mode="lines",
        line=dict(color=TEXT_PRIMARY, width=2, dash="dot"), showlegend=False,
        hovertemplate=f"{metodo}<br>pendiente: {m:.2f} %/año<br>r = {r:.2f}<extra></extra>",
    ), row=1, col=i + 1)
    fig.add_annotation(x=0.5, y=0.04, xref="x domain", yref="y domain",
                       text=f"r = {r:.2f}", showarrow=False,
                       font=dict(size=12, color=TEXT_SECOND), row=1, col=i + 1)

fig.update_annotations(font=dict(size=13, color=TEXT_PRIMARY))
fig.update_xaxes(title_text="Años en operación")
fig.update_yaxes(title_text="Corte de agua (%)", col=1)
titular(fig, "El corte de agua lo explica la antigüedad, no el sistema de levantamiento",
        "Cada panel repite la misma relación; los métodos solo ocupan tramos distintos del eje del tiempo",
        alto=480)
fig.show()

**Lectura.** La pendiente es positiva y la correlación fuerte **dentro de cada método** (r entre 0.79 y
0.87): el corte de agua sube de forma sostenida con los años de operación, y la pendiente es prácticamente
idéntica en los cuatro grupos: **3.1-3.4 puntos porcentuales por año**. La diferencia entre métodos del gráfico anterior se explica casi por
completo por dónde cae cada grupo en el eje horizontal: BES concentra pozos jóvenes, Flujo Natural los
veteranos. En otras palabras, **el método de extracción no cambia la física del influjo de agua**;
lo que cambia es la población de pozos a la que se aplica.

In [48]:
fig = go.Figure()
for estado in orden_estado:
    sub = df[df["estado_operativo"] == estado]
    fig.add_trace(go.Scatter(
        x=sub["corte_agua_pct"], y=sub["produccion_bpd"], mode="markers", name=estado,
        marker=dict(color=COLOR_ESTADO[estado], size=10, opacity=0.8,
                    line=dict(color=SURFACE, width=1.5)),
        customdata=sub[["id_pozo", "metodo_extraccion", "anios_operacion"]],
        hovertemplate=("%{customdata[0]} · %{customdata[1]}<br>Corte de agua: %{x:.1f}%"
                       "<br>Producción: %{y:.1f} bpd<br>Antigüedad: %{customdata[2]:.1f} años<extra></extra>"),
    ))

# Tendencia ajustada unicamente sobre los pozos activos
m, b = np.polyfit(activos["corte_agua_pct"], activos["produccion_bpd"], 1)
xs = np.array([activos["corte_agua_pct"].min(), activos["corte_agua_pct"].max()])
fig.add_trace(go.Scatter(x=xs, y=m * xs + b, mode="lines", name="Tendencia (activos)",
                         line=dict(color=TEXT_PRIMARY, width=2, dash="dot"),
                         hovertemplate=f"pendiente: {m:.1f} bpd por punto de corte de agua<extra></extra>"))

r_act = activos["corte_agua_pct"].corr(activos["produccion_bpd"])
fig.update_xaxes(title_text="Corte de agua (%)")
fig.update_yaxes(title_text="Producción (bpd)")
titular(fig, "Producción frente a corte de agua",
        f"Solo entre pozos activos la relación es real: r = {r_act:.2f} · cerrados y en mantenimiento "
        "se muestran para no ocultar un tercio del inventario", alto=540)
fig.show()

**Lectura.** Entre los pozos **activos** la relación es claramente descendente (r ≈ −0.46): más agua,
menos crudo. Las dos bandas planas de la parte inferior no son parte de esa tendencia y por eso están
codificadas con color propio: los **cerrados** están todos en 0 bpd por definición y los pozos **en
mantenimiento** operan por debajo de ~45 bpd independientemente de su corte de agua. Mezclarlos con
los activos en un solo ajuste produciría una correlación artificialmente fuerte; separarlos es lo que
permite leer el efecto real del agua sobre el caudal.

In [49]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=activos["presion_cabeza_psi"], y=activos["produccion_bpd"], mode="markers",
    marker=dict(color=CAT[0], size=10, opacity=0.8, line=dict(color=SURFACE, width=1.5)),
    customdata=activos[["id_pozo", "corte_agua_pct", "anios_operacion"]], showlegend=False,
    hovertemplate=("%{customdata[0]}<br>Presión de cabeza: %{x:.0f} psi<br>Producción: %{y:.1f} bpd"
                   "<br>Corte de agua: %{customdata[1]:.1f}%<extra></extra>"),
))
m, b = np.polyfit(activos["presion_cabeza_psi"], activos["produccion_bpd"], 1)
xs = np.array([activos["presion_cabeza_psi"].min(), activos["presion_cabeza_psi"].max()])
fig.add_trace(go.Scatter(x=xs, y=m * xs + b, mode="lines",
                         line=dict(color=TEXT_PRIMARY, width=2, dash="dot"), showlegend=False,
                         hovertemplate="tendencia<extra></extra>"))

r_p = activos["presion_cabeza_psi"].corr(activos["produccion_bpd"])
fig.update_xaxes(title_text="Presión de cabeza (psi)")
fig.update_yaxes(title_text="Producción (bpd)")
titular(fig, "Presión de cabeza frente a producción — solo pozos activos",
        f"r = {r_p:.2f}: la energía remanente del yacimiento sigue siendo el mejor predictor positivo del caudal")
fig.show()

## 8. Matriz de correlaciones

El paso anterior examinó relaciones de a dos. La matriz las revisa todas a la vez y, sobre todo,
señala **qué variables no se relacionan con nada** —información igual de útil para decidir qué medir y
qué modelar. Se calcula sobre los **pozos activos**: incluir los cerrados (producción idénticamente 0)
distorsionaría cualquier coeficiente en el que participe el caudal.

In [50]:
cols_num = ["anios_operacion", "corte_agua_pct", "produccion_bpd", "produccion_acumulada_mbbl",
            "presion_cabeza_psi", "api_gravedad", "profundidad_m", "temperatura_fondo_c"]
etiquetas = ["Años en operación", "Corte de agua", "Producción (bpd)", "Producción acumulada",
             "Presión de cabeza", "Gravedad API", "Profundidad", "Temperatura de fondo"]

corr = activos[cols_num].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=etiquetas, y=etiquetas,
    colorscale=DIVERGING, zmid=0, zmin=-1, zmax=1,
    xgap=2, ygap=2,
    text=corr.values, texttemplate="%{text:.2f}",
    textfont=dict(size=12),
    colorbar=dict(title=dict(text="r de Pearson", font=dict(size=12)), outlinewidth=0,
                  tickvals=[-1, -0.5, 0, 0.5, 1], len=0.85),
    hovertemplate="%{y}<br>%{x}<br>r = %{z:.2f}<extra></extra>",
))
fig.update_xaxes(tickangle=-35, showgrid=False)
fig.update_yaxes(autorange="reversed", showgrid=False)
titular(fig, "Matriz de correlaciones — pozos activos",
        "Azul = relación inversa · gris = sin relación · rojo = relación directa", alto=620)
fig.update_layout(margin=dict(l=150, r=30, t=100, b=140))
fig.show()

In [52]:
# Pares más correlacionados, en orden de intensidad
pares = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().rename("r").reset_index()
             .rename(columns={"level_0": "variable_1", "level_1": "variable_2"}))
pares["|r|"] = pares["r"].abs()
pares.sort_values("|r|", ascending=False).head(8).round(2).reset_index(drop=True)

,variable_1,variable_2,r,|r|
0,anios_operacion,corte_agua_pct,0.87,0.87
1,anios_operacion,presion_cabeza_psi,-0.76,0.76
2,corte_agua_pct,presion_cabeza_psi,-0.65,0.65
3,anios_operacion,produccion_bpd,-0.54,0.54
4,corte_agua_pct,produccion_bpd,-0.46,0.46
5,anios_operacion,produccion_acumulada_mbbl,0.41,0.41
6,produccion_bpd,presion_cabeza_psi,0.41,0.41
7,produccion_acumulada_mbbl,presion_cabeza_psi,-0.38,0.38


**Lectura.**

- **Antigüedad ↔ corte de agua (r ≈ 0.87)** es, con diferencia, la relación dominante del campo: la
  variable más predecible del inventario es cuánta agua produce un pozo, y basta su fecha de inicio
  para estimarla.
- **Corte de agua ↔ presión de cabeza (r ≈ −0.65)** y **presión ↔ producción (r ≈ 0.41)**: la cadena
  física es coherente —el pozo pierde energía, entra agua, cae el caudal—; las tres variables miden
  facetas del mismo proceso de agotamiento y **no deben tratarse como predictores independientes** en
  un modelo.
- **Producción acumulada ↔ antigüedad (r ≈ 0.4)**: los pozos viejos han producido más en total aunque
  hoy produzcan menos por día. Distinguir caudal instantáneo de volumen histórico evita la
  contradicción aparente.
- **Profundidad, temperatura de fondo y gravedad API no se correlacionan con nada** (|r| < 0.15). Son
  propiedades de la roca y del fluido, no del estado operativo del pozo. La API, eso sí, no es ruido:
  su variación es **espacial**, y eso es lo que muestra la sección 10.

## 9. Método de extracción a lo largo del tiempo

Si el método está atado a la madurez del pozo, la composición por cohorte de arranque debería mostrar
un relevo tecnológico. Se grafica la **participación porcentual** dentro de cada cohorte, porque las
cohortes no tienen el mismo número de pozos y las cifras absolutas confundirían tamaño con tendencia.

In [21]:
comp = (pd.crosstab(df["cohorte_inicio"], df["metodo_extraccion"], normalize="index") * 100).round(1)
conteo_cohorte = df["cohorte_inicio"].value_counts().reindex(comp.index)

fig = go.Figure()
for metodo in orden_metodo:
    fig.add_trace(go.Bar(
        x=comp.index.astype(str), y=comp[metodo], name=metodo,
        marker=dict(color=COLOR_METODO[metodo], line=dict(color=SURFACE, width=2)),
        text=[f"{v:.0f}%" if v >= 8 else "" for v in comp[metodo]],
        textposition="inside", insidetextfont=dict(color=SURFACE, size=12),
        hovertemplate=f"{metodo}<br>%{{x}}: %{{y:.1f}}%<extra></extra>",
    ))

fig.update_layout(barmode="stack", bargap=0.35)
fig.update_yaxes(title_text="Participación dentro de la cohorte (%)", range=[0, 100])
fig.update_xaxes(title_text="Cohorte de inicio de operación",
                 ticktext=[f"{c}<br><span style='font-size:11px'>n={n}</span>"
                           for c, n in zip(comp.index.astype(str), conteo_cohorte)],
                 tickvals=list(comp.index.astype(str)))
titular(fig, "Relevo tecnológico: cómo cambió el método de completamiento por cohorte",
        "El Flujo Natural pasa del 37 % de los pozos de 2008-2011 a menos del 5 % de los de 2020-2023", alto=520)
fig.show()

In [22]:
fig = go.Figure()
for metodo in orden_metodo:
    sub = df[df["metodo_extraccion"] == metodo]
    conteo = sub["anio_inicio"].value_counts().sort_index()
    conteo = conteo.reindex(range(df["anio_inicio"].min(), df["anio_inicio"].max() + 1), fill_value=0)
    fig.add_trace(go.Scatter(
        x=conteo.index, y=conteo.rolling(3, center=True, min_periods=1).mean(),
        mode="lines+markers", name=metodo,
        line=dict(color=COLOR_METODO[metodo], width=2),
        marker=dict(size=8, color=COLOR_METODO[metodo], line=dict(color=SURFACE, width=1.5)),
        hovertemplate=f"{metodo}<br>%{{x}}: %{{y:.1f}} pozos/año (media móvil 3a)<extra></extra>",
    ))

fig.update_xaxes(title_text="Año de inicio de operación", dtick=2)
fig.update_yaxes(title_text="Pozos perforados por año (media móvil de 3 años)")
titular(fig, "Pozos puestos en producción por año y método",
        "La BES desplaza progresivamente al Flujo Natural y al Bombeo Mecánico")
fig.show()

**Lectura.** El relevo es inequívoco. En la cohorte **2008-2011** el Flujo Natural representa el 37 %
de los completamientos y la BES apenas el 12 %; en la cohorte **2020-2023** la BES
es el método dominante con el 46 % de los pozos nuevos y el Flujo Natural cae al 4.6 %. Es el patrón esperable en un
campo de crudo pesado que envejece: al caer la energía natural del yacimiento, el levantamiento
artificial de mayor capacidad deja de ser una opción y pasa a ser un requisito. Esto cierra el
argumento de la sección 6: **el método no es una causa del corte de agua sino un marcador de la época
—y por tanto de la madurez— de cada pozo**.

## 10. Gravedad API por zona operativa

La API no se correlacionaba con ninguna variable operativa. La hipótesis es que su variación es
geográfica: la calidad del crudo depende del sector del campo, no de cómo se opere el pozo.

In [23]:
fig = go.Figure()
for i, zona in enumerate(orden_zona):
    sub = df[df["zona_operativa"] == zona]
    fig.add_trace(go.Box(
        y=sub["api_gravedad"], name=zona,
        marker=dict(color=SEQ[i], size=5, opacity=0.5, line=dict(color=SURFACE, width=1)),
        line=dict(width=2), fillcolor="rgba(0,0,0,0)",
        boxpoints="all", jitter=0.4, pointpos=0, width=0.55,
        hovertemplate="%{x}<br>API: %{y:.1f}°<extra></extra>",
    ))

for i, zona in enumerate(orden_zona):
    mediana = df.loc[df["zona_operativa"] == zona, "api_gravedad"].median()
    fig.add_annotation(x=zona, y=df["api_gravedad"].max() + 0.9, text=f"{mediana:.1f}°",
                       showarrow=False, font=dict(size=12, color=TEXT_PRIMARY))

fig.update_yaxes(title_text="Gravedad API (°API)")
fig.update_xaxes(title_text="Zona operativa")
fig.update_layout(showlegend=False)
titular(fig, "Gravedad API por zona operativa",
        "Gradiente de ~5 °API entre Centro y Oriente · el número sobre cada caja es la mediana de la zona",
        alto=520)
fig.show()

In [24]:
(df.groupby("zona_operativa", observed=True)
   .agg(pozos=("id_pozo", "count"),
        api_media=("api_gravedad", "mean"),
        api_desv=("api_gravedad", "std"),
        api_min=("api_gravedad", "min"),
        api_max=("api_gravedad", "max"),
        corte_agua_medio=("corte_agua_pct", "mean"),
        produccion_media=("produccion_bpd", "mean"))
   .round(2))

,pozos,api_media,api_desv,api_min,api_max,corte_agua_medio,produccion_media
zona_operativa,,,,,,,
Centro,33,9.30,1.14,7.5,12.1,44.53,122.94
Norte,50,10.73,1.25,8.2,13.5,45.81,102.62
Occidente,50,12.37,1.37,8.6,14.8,41.99,134.23
Sur,57,13.49,1.25,10.8,15.8,47.31,129.81
Oriente,60,14.45,1.50,11.7,18.7,47.14,108.12


**Lectura.** La hipótesis se confirma y de forma bastante limpia. Las zonas se ordenan en un gradiente
continuo de medias —**Centro 9.3 °API → Norte 10.7 → Occidente 12.4 → Sur 13.5 → Oriente 14.5**— y la
dispersión *dentro* de cada zona es estrecha (desviación de apenas 1.1-1.5 °API) frente a un rango
total de 5 puntos entre extremos. Es decir: **saber en qué zona está un pozo predice la calidad de su
crudo mucho mejor que cualquier variable operativa**. El Centro produce el crudo más pesado del campo,
con implicaciones directas en dilución, viscosidad y costo de transporte; el Oriente, el más liviano
dentro de un rango que sigue siendo íntegramente de crudo pesado.

## 11. Estado operativo y mantenimiento

Cierre operativo: cómo se reparte el inventario entre estados y si la carga de mantenimiento se
concentra en algún método.

In [25]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.18,
                    subplot_titles=("Pozos por estado operativo",
                                    "Pozos que requieren mantenimiento, por método"))

est = df["estado_operativo"].value_counts().reindex(orden_estado)
fig.add_trace(go.Bar(
    x=est.index.astype(str), y=est.values,
    marker=dict(color=[COLOR_ESTADO[e] for e in est.index], line=dict(color=SURFACE, width=2)),
    text=[f"{v} ({v/len(df):.0%})" for v in est.values], textposition="outside",
    textfont=dict(color=TEXT_SECOND, size=12), showlegend=False,
    hovertemplate="%{x}<br>%{y} pozos<extra></extra>",
), row=1, col=1)

mant = (df.groupby("metodo_extraccion", observed=True)["requiere_mantenimiento"].mean() * 100).round(1)
fig.add_trace(go.Bar(
    x=mant.index.astype(str), y=mant.values,
    marker=dict(color=[COLOR_METODO[m] for m in mant.index], line=dict(color=SURFACE, width=2)),
    text=[f"{v:.0f}%" for v in mant.values], textposition="outside",
    textfont=dict(color=TEXT_SECOND, size=12), showlegend=False,
    hovertemplate="%{x}<br>%{y:.1f}% requiere mantenimiento<extra></extra>",
), row=1, col=2)

fig.update_annotations(font=dict(size=13, color=TEXT_PRIMARY))
fig.update_yaxes(title_text="Nº de pozos", range=[0, est.max() * 1.2], row=1, col=1)
fig.update_yaxes(title_text="% del método", range=[0, 60], row=1, col=2)
fig.update_layout(bargap=0.4)
titular(fig, "Situación operativa del inventario", alto=480)
fig.show()

**Lectura.** El **74 % del inventario está activo**, un 18 % en mantenimiento y un 8 % cerrado. La
necesidad de intervención es alta y bastante pareja entre métodos (37-48 %), con el Flujo Natural
ligeramente arriba y el Bombeo Mecánico abajo; la diferencia es pequeña frente al tamaño de los grupos,
así que no sostiene por sí sola una conclusión sobre confiabilidad de la tecnología. Lo que sí es
accionable es el volumen: **104 pozos con mantenimiento pendiente** es una cola de trabajo que conviene
priorizar por producción en riesgo, no por método.

## 12. Hallazgos

1. **El dataset está limpio.** 250 pozos, sin nulos ni duplicados, sin valores fuera de rango físico.
   El trabajo de preparación fue de estandarización (fechas, categóricas ordenadas, variables
   derivadas), no de reparación. Dos columnas son prescindibles: `campo` es constante y `activo` es
   redundante con `estado_operativo`.

2. **La antigüedad gobierna el campo.** La correlación entre años en operación y corte de agua es
   **r ≈ 0.87**, la más fuerte del dataset con amplia diferencia, y se sostiene dentro de cada método
   de extracción. El agua avanza a un ritmo de 3.1-3.4 puntos porcentuales por año, estable en todos los métodos.

3. **El método de extracción no causa el corte de agua: lo acompaña.** La diferencia aparente entre
   Flujo Natural (mediana ~55 % de agua) y BES (~40 %) se explica por la antigüedad de cada grupo
   (~15 años frente a ~7.5). Tratar el método como predictor del corte de agua sería confundir el
   marcador con la causa.

4. **Existe un relevo tecnológico documentado en los datos.** La BES pasa del 12 % de los
   completamientos en la cohorte 2008-2011 al 46 % en 2020-2023, mientras el Flujo Natural
   casi desaparece: la respuesta esperable a la pérdida de energía del yacimiento.

5. **La cadena de agotamiento es coherente y redundante.** Antigüedad, corte de agua, presión de cabeza
   y producción están fuertemente entrelazadas (|r| entre 0.4 y 0.87). Para modelado predictivo son
   facetas del mismo fenómeno y no deben entrar como variables independientes sin tratamiento.

6. **La calidad del crudo es un asunto geográfico.** La gravedad API no se correlaciona con ninguna
   variable operativa, pero segrega nítidamente por zona: gradiente continuo de ~9.3 °API en el Centro
   a ~14.5 °API en el Oriente, con dispersión interna estrecha. Todo el campo es crudo pesado o
   extrapesado.

7. **Profundidad y temperatura de fondo no discriminan nada** en este inventario (|r| < 0.11 con toda
   variable operativa): el campo ataca un intervalo homogéneo.

### Qué haría falta para ir más lejos

El dataset es una **fotografía**: un registro por pozo con su último reporte. Las conclusiones sobre
declinación se infieren comparando pozos de distinta edad, no siguiendo a cada pozo en el tiempo. Una
**serie histórica de producción y corte de agua por pozo** permitiría ajustar curvas de declinación
reales, separar el efecto de las intervenciones y estimar reservas remanentes — cosas que con esta
estructura de datos solo pueden aproximarse.